In [1]:
import jax.numpy as jnp
import roughpy_jax as rpj
import numpy as np
import jax
from roughpy_jax.streams import LieIncrementStream
from roughpy_jax.streams.lie_increment_stream import _zero_lie
from roughpy_jax.intervals import IntervalType, Partition
from roughpy_jax.streams.piecewise_abelian_stream import to_piecewise_abelian_stream
from roughpy_jax.algebra import to_signature, antipode, to_log_signature, lie_to_tensor, as_free_tensor
from roughpy_jax.dense_algebra import get_batch_shape, _algebra_scalar_multiply, broadcast_to_batch_shape

In [38]:
times = [[0., 0.5, 1.0], [0.,0.5, 1.0], [0.,0.5, 1.0], [0.,0.5, 1.0]]
data = [jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=jnp.float32), 
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32), 
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32),
        jnp.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]], dtype=jnp.float32)]
Lie_Basis = rpj.LieBasis(width=2, depth=2)
Tensor_Basis = rpj.to_tensor_basis(Lie_Basis)
X_Lie_test = LieIncrementStream.from_increments(timestamps=times, data=data, input_data_basis=None, resolution=2, lie_basis=Lie_Basis)

In [ ]:
# fix this later
def make_data_liestream(times, data, R, Lie_Basis):
    ''' 
    times must be of the shape (B, N) where B is batch size and N is the length of the time series
    data must of the shape (B, N, D) where D is the dimension of the space it takes values in
    '''
    timestamps = [time[1:] for time in times]
    data_inc = [jnp.diff(datum, axis=1) for datum in data]
    return LieIncrementStream.from_increments(timestamps=timestamps, data=data_inc, input_data_basis=None, resolution=R, lie_basis=Lie_Basis)

In [37]:
X_Lie.signature().shape

(4, 7)

In [ ]:
# series_to_stream cannot be combined with vmap since the creation of a LieIncrementStream is incompatible

def series_to_stream(X, times, R, lie_basis):
    '''
    X, array - N by W time series where N is the no. of timesteps, W is the width (dimension) of the space X takes values in
    times, array - one-dimensional array with timing of each event in X
    R, integer - resolution of LieIncrementStream
    '''
    X_inc = jnp.diff(X, axis=0)
    timestamps = jnp.delete(times, -1)
    X_Lie = LieIncrementStream.from_increments(
                    timestamps=timestamps,
                    data=X_inc,
                    input_data_basis=None,
                    resolution=R,
                    lie_basis=lie_basis)
    return X_Lie

def batch_series_to_stream(X_batch, times, R, lie_basis):
    return [series_to_stream(X, t, 2, lie_basis) for X, t in zip(X_batch, times)]

In [4]:
# calculate signatures and log-signatures over each interval in a specified partition
# LieIncrementStream not (yet) JAX , so manually batching instead

def logsigs_over_intervals(X_Lie, intervals):
    '''
    X_Lie - LieIncrementStream class element (Roughpy-jax)                                                      
    partition - Interval class element (Roughpy-jax). To construct an interval first construct a partition, using 
    Partition(endpoints, IntervalType.ClOpen) where endpoints is an array of values between 0 and 1 which give the endpoints of the interval in the partition
    '''

    X_LSP = tuple(lie_to_tensor(X_Lie.log_signature(interval)) for interval in intervals) 
    
    return X_LSP

def batch_logsigs_over_intervals(X_Lie_batch, intervals_batch):
    '''
    X_Lie_batch - list of B LieIncrementStreams
    partition_batch - list of B paritions 
    '''
    return [logsigs_over_intervals(x_lie, p) for x_lie, p in zip(X_Lie_batch, intervals_batch)]

def sigs_from_logsigs(X_LSP):
    X_SP = tuple(rpj.ft_exp(x_ls, out_basis=x_ls.basis) for x_ls in X_LSP)
    return X_SP

def batch_sigs_from_logsigs(X_LSP_batch):
    return [sigs_from_logsigs(x_lsp) for x_lsp in X_LSP_batch]

In [5]:
# helper function to create `interval_count` uniform intervals from 0 to 1

def uniform_intervals(interval_count):
    endpoints = jnp.linspace(0, 1, interval_count + 1, dtype=jnp.float32).tolist()
    partition = Partition(endpoints, IntervalType.ClOpen)
    return partition.to_intervals()

In [58]:
def logsigs_over_intervals(X_Lie, intervals):
    return tuple(
        lie_to_tensor(X_Lie.log_signature(interval))
        for interval in intervals
    )


def sigs_from_logsigs(X_LSP):
    return tuple(rpj.ft_exp(x_ls, out_basis=x_ls.basis) for x_ls in X_LSP)

def batch_sigs_over_intervals(X_Lie_batch, intervals_batch):

    X_LSP_batch = [logsigs_over_intervals(x_lie, intervals) for x_lie, intervals in zip(X_Lie_batch, intervals_batch)]
    print(X_LSP_batch[0][0].shape)
    X_SP_batch = [sigs_from_logsigs(x_lsp) for x_lsp in X_LSP_batch]
    return jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*X_LSP_batch), jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*X_SP_batch)


In [65]:
tuple(jnp.stack([lie_to_tensor(x_lie.log_signature(interval)) for x_lie in data_lie]) for interval in intervals[0]) 

TypeError: stack requires ndarray or scalar arguments, got <class 'roughpy_jax.algebra.DenseFreeTensor'> at position 0.

In [61]:
intervals = tuple(uniform_intervals(4) for i in range(B))
X_LSP_batch_test, X_SP_batch_test = batch_sigs_over_intervals(data_lie, intervals)
X_LSP_batch_test # (3,1,7) but we want (3,7)

(1, 7)


(<roughpy_jax.algebra.DenseFreeTensor at 0x1ad53d590a0>,
 <roughpy_jax.algebra.DenseFreeTensor at 0x1ad51923740>)

In [7]:
intervals = tuple(uniform_intervals(4) for i in range(B))
X_LSP_batch = batch_logsigs_over_intervals(data_lie, intervals)
X_SP_batch = batch_sigs_from_logsigs(X_LSP_batch)

In [8]:
# truncating the log-signatures to depth n-1, we then change depth back to n, so that calculations still work (but the nth layer is now all zero)

def trunc(X_LSP, old_depth, new_depth):
    return tuple(x.change_depth(new_depth).change_depth(old_depth) for x in X_LSP)

def batch_truncate(X_LSP_batch, old_depth, new_depth):
    return [trunc(X_LSP, old_depth, new_depth) for X_LSP in X_LSP_batch]

In [ ]:
# making the first one in each signature a zero 

def one_to_zero(X_SP, tensor_basis):

    X_sub = []
    
    for x in X_SP:
        x = jnp.array(x.__array__())
        x.at[0].set(0)
        x = rpj.FreeTensor(x, tensor_basis)
        X_sub.append(x)
        
    return X_sub

def batch_one_to_zero(X_SP_batch, tensor_basis):
    return [one_to_zero(X_SP, tensor_basis) for X_SP in X_SP_batch]

In [ ]:
rpj.FreeTensor.zero(basis=Tensor_Basis, batch_dims=(3,)).shape # we want (P, B, tensor_dim) so can make a tuple over (B, D) tensors

(3, 7)

In [ ]:
# setting up initial conditions for the PDE - ADD ABILITY TO BATCH (COMPUTING GRAM MATRIX ALL IN ONE AND STUFF)

def initialise_PDE(X_SP, Y_SP, X_SPT_zero, Y_SPT_zero, n, Tensor_Basis):
    '''
    X_SP 
    '''
    L = len(X_SP)
    M = len(Y_SP)
    
    # K represents our target function f in the original PDE (Algorithm 5.1)
    K = jnp.zeros((L+1, M+1), dtype=jnp.float32) 

    # phi and psi follow notation from Algorithm 5.1 - batch_dims set to (1,) for now

    zero_tensor = rpj.FreeTensor.zero(basis=Tensor_Basis, batch_dims=(1,))
    phi = [[zero_tensor]*(M+1)]*(L+1)    
    psi = [[zero_tensor]*(M+1)]*(L+1)

    # Since the paper gives K[0, v] as the inner product of Z_0^x and Z_v^y and analogous for K[u, 0], I assume we are setting Z_0^x = 1 = Z_0^y where 1 = (1,0,0,0,...) 
    # is in the signature sense
 
    K = K.at[0, :].set(1)
    K = K.at[:, 0].set(1)

    for i in range(1, L + 1):
        phi[i][0] = X_SPT_zero[i-1]
        
    for j in range(1, M + 1):
        psi[0][j] = Y_SPT_zero.__array__()[j-1]        
    return K, phi, psi 

In [11]:
# Helper functions used to help calculations present in algorithm 5.1

def inner_prod(X,Y): 
    return jnp.sum(X.__array__()*Y.__array__())  

def right_adj(A,C, tensor_basis):

    ant_A = antipode(A)
    ant_C = antipode(C)
    
    left_adj = rpj.ft_adjoint_left_mul(ant_A, ant_C)

    left_adj = rpj.FreeTensor(left_adj, basis=tensor_basis)  
    
    adj_A_C = antipode(left_adj)
    
    return adj_A_C

    
def eval_adj(phi, psi, x, y, tensor_basis):
    
    r_x_y = right_adj(x, y, tensor_basis)  
    r_y_x = right_adj(y, x, tensor_basis)

    return inner_prod(phi, r_x_y) + inner_prod(psi, r_y_x)

def add_tensor_scalar(a, s):
    cls = type(a)
    scalar = jnp.asarray(s)
    ext_scalar = broadcast_to_batch_shape(scalar, a.batch_shape)
    result_data = jnp.add(a.data, ext_scalar)
    return cls(result_data, a.basis)

In [12]:
def algorithm_solve(X_LSP, Y_LSP, X_LSPT, Y_LSPT, K, phi, psi, Tensor_Basis):

    L = len(X_LSP)
    M = len(Y_LSP)

    for i in range(L):
        for j in range(M):
                
            # phi
            phi[i+1][j+1] =  phi[i][j+1] + X_LSPT[i].__mul__(K[i,j])\
                            + rpj.ft_mul(phi[i][j+1], X_LSPT[i])\
                            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(psi[i][j+1], X_LSP[i])), -inner_prod(psi[i][j+1], X_LSPT[i]))

            # psi
            psi[i+1][j+1] =  psi[i+1][j] + Y_LSPT[j].__mul__(K[i,j])\
                            + rpj.ft_mul(psi[i+1][j],Y_LSPT[j])\
                            + add_tensor_scalar(as_free_tensor(rpj.ft_adjoint_left_mul(phi[i+1][j], Y_LSP[j])), -inner_prod(phi[i+1][j], Y_LSPT[j]))

            eval_adj_ = eval_adj(phi[i][j], psi[i][j], X_LSP[i], Y_LSP[j], Tensor_Basis)
                

            # the kernel equation
            next_eval_adj = eval_adj(phi[i+1][j+1], psi[i+1][j+1], X_LSP[i], Y_LSP[j], Tensor_Basis)
            temp_2 =  eval_adj(phi[i][j+1], psi[i][j+1], X_LSP[i], Y_LSP[j], Tensor_Basis)
            temp_3 = eval_adj(phi[i+1][j], psi[i+1][j], X_LSP[i], Y_LSP[j], Tensor_Basis)
    
            G = inner_prod(X_LSP[i],Y_LSP[j])
            f_1 = K[i,j] * G + eval_adj_
            f_2 = K[i,j+1] * G + temp_2 
            f_3 = K[i+1,j] * G + temp_3 

            u_p = K[i+1,j] + K[i,j+1] - K[i,j] + f_1
            f_p = u_p * G + next_eval_adj


            K[i+1,j+1] =  K[i+1,j] + K[i,j+1] - K[i,j] + (1./4)*(f_1 + f_2 + f_3 + f_p)
    return K, phi, psi

In [13]:
def solve_pair(X_SP, Y_SP, X_LSP, Y_LSP, X_LSPT, Y_LSPT, X_SPT_zero, Y_SPT_zero, n, Tensor_Basis):

    # Initialise the PDE state for this pair
    K_pair, phi_pair, psi_pair = initialise_PDE(X_SP, Y_SP, X_SPT_zero, Y_SPT_zero, n, Tensor_Basis)

    # Solve the PDE for this pair
    K_pair, phi_pair, psi_pair = algorithm_solve(X_LSP, Y_LSP, X_LSPT, Y_LSPT, K_pair, phi_pair, psi_pair, Tensor_Basis)

    # Return the final kernel value for this pair
    return K_pair[-1, -1]

solve_pairs = jax.jit(jax.vmap(solve_pair, in_axes=(0, 0, 0, 0, 0, 0, 0, 0, None, None)))

In [14]:
n = 2
X_SPT_batch = batch_truncate(X_SP_batch, n, n-1)
X_LSPT_batch = batch_truncate(X_LSP_batch, n, n-1)
X_SPT_zero_batch = batch_one_to_zero(X_SPT_batch, Tensor_Basis)

In [15]:
B = len(X_SP_batch)
pairs = jnp.stack(jnp.triu_indices(B), axis=1)

In [18]:
X_SP = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*[jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*batch) for batch in X_SP_batch])
X_LSP = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*[jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*batch) for batch in X_LSP_batch])
X_LSPT = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*[jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*batch) for batch in X_SP_batch])
X_SPT_zero_batch = jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*[jax.tree.map(lambda *xs: jnp.stack(xs, axis=0),*batch) for batch in X_SPT_zero_batch])

In [ ]:
initialise_PDE()

In [108]:
X_SP.shape # (B, P, ...)

(3, 4, 1, 7)

In [111]:
pairs[:, 0], pairs[:, 1]

(Array([0, 0, 0, 1, 1, 2], dtype=int32),
 Array([0, 1, 2, 1, 2, 2], dtype=int32))

In [19]:
batch_i = pairs[:, 0]
batch_j = pairs[:, 1]

X_i = jax.tree.map(lambda x: x[batch_i], X_SP_batch)
X_j = jax.tree.map(lambda x: x[batch_j], X_SP_batch)

X_LSP_i = jax.tree.map(lambda x: x[batch_i], X_LSP_batch)
X_LSP_j = jax.tree.map(lambda x: x[batch_j], X_LSP_batch)

X_LSPT_i = jax.tree.map(lambda x: x[batch_i], X_LSPT_batch)
X_LSPT_j = jax.tree.map(lambda x: x[batch_j], X_LSPT_batch)

X_SPT_zero_i = jax.tree.map(lambda x: x[batch_i], X_SPT_zero_batch)
X_SPT_zero_j = jax.tree.map(lambda x: x[batch_j], X_SPT_zero_batch)

In [30]:
pair_values = solve_pairs(
    X_i,
    X_j,
    X_LSP_i,
    X_LSP_j,
    X_LSPT_i,
    X_LSPT_j,
    X_SPT_zero_i,
    X_SPT_zero_j,
    n,
    Tensor_Basis,
)

(4, 1, 7)


TypeError: 'DenseFreeTensor' object is not subscriptable